# Geometry-Aware DPO Training on Google Colab
**Runtime**: GPU → T4  
**预计时间**: SFT ~1h，DPO baseline ~2h，GA-DPO ~2h

In [ ]:
# ① 挂载 Google Drive（checkpoint 存这里，断线不丢）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ② 上传项目 zip 到 Drive 后，解压到 /content
import zipfile, os

zip_path = '/content/drive/MyDrive/geometry-aware-dpo.zip'   # 改成你的实际路径
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')

os.chdir('/content/geometry-aware-dpo')
print('Working dir:', os.getcwd())

In [ ]:
# ③ 安装依赖
!pip install -q \
    transformers datasets \
    'trl>=1.4.0' \
    peft bitsandbytes accelerate \
    pyyaml einops

# 验证 GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ④ 下载 Qwen2.5-1.5B-Instruct
# 如果之前已下载到 Drive，直接解压更快（见注释）
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='Qwen/Qwen2.5-1.5B-Instruct',
    local_dir='/content/qwen1_5b',
    ignore_patterns=['*.msgpack', '*.h5'],
)
print('Model downloaded to /content/qwen1_5b')

# 如果已经存在 Drive 里，用这行代替上面（更快）：
# !cp -r /content/drive/MyDrive/qwen1_5b /content/qwen1_5b

In [ ]:
# ⑤ 建输出目录
!mkdir -p /content/drive/MyDrive/gadpo_outputs

In [ ]:
# ⑥ SFT 训练（~1 小时）
!python scripts/train_sft.py --config configs/sft_qwen_1_5b.yaml

In [ ]:
# ⑦ DPO Baseline 训练（~2 小时）
!python scripts/train_dpo_baseline.py \
    --config configs/dpo_baseline_qwen_1_5b.yaml \
    --sft_checkpoint /content/drive/MyDrive/gadpo_outputs/sft_qwen_1_5b/final

In [ ]:
# ⑧ GA-DPO 训练（~2 小时）
!python scripts/train_gadpo.py \
    --config configs/gadpo_qwen_1_5b.yaml \
    --sft_checkpoint /content/drive/MyDrive/gadpo_outputs/sft_qwen_1_5b/final

In [ ]:
# ⑨ 评测对比
!python scripts/evaluate.py \
    --config configs/gadpo_qwen_1_5b.yaml \
    --baseline /content/drive/MyDrive/gadpo_outputs/dpo_baseline_qwen_1_5b/final \
    --gadpo /content/drive/MyDrive/gadpo_outputs/gadpo_qwen_1_5b/final \
    --output /content/drive/MyDrive/gadpo_outputs/eval_results_1_5b.json